# SongUNet covariance Tikhonov across training-set size

**Question.** Does the scale-selective effect seen in
`songunet_covariance_tikhonov.ipynb` at $n_{\mathrm{train}}=2$ persist as the training set grows?
This is the dataset-size follow-up requested by Prof. Baptista.

**Fixed covariance weight.** Every covariance arm uses the same analytic Matérn spectrum
$\lambda_{\mathrm{pop}}(k)$. In the completed 25-arm experiment, estimates from 2, 8, 16, or
32 fields and the analytic spectrum gave the same qualitative result, with no empirical pool
size consistently best. The analytic spectrum is therefore the cleanest intervention: it is
independent of $n_{\mathrm{train}}$, independent of the realized training fields, and uses no
extra data. Its inverse is budget-normalized exactly as in the source notebook.

**Design.**

- Nested training sets: $n_{\mathrm{train}}\in\{2,4,8,16,32\}$ from the same 200-field,
  seed-42 pool; the single rescaling factor is the one fixed by the first two fields in the
  completed experiment.
- Exact SongUNet/EDMPrecond configuration from the completed run: `model_channels=16`,
  880,097 parameters, `sigma_data=0.5`, training seed 0.
- At every $n$: unregularized control; isotropic Tikhonov at
  $c\in\{0.003,0.01,0.03,0.1\}$; analytic covariance Tikhonov at the same four $c$ values.
  This is 45 cells conceptually. The 9 completed $n=2$ artifacts are reused, so 36 new arms
  remain.
- **Fixed 100,000 optimizer updates per arm.** This exactly matches the completed $n=2$ run
  (50,000 epochs x 2 batch-1 updates). Holding 50,000 epochs at every $n$ would silently give
  $n=32$ sixteen times as many updates as $n=2$, confounding dataset size with optimization
  budget. Batch size remains 1, matching Baptista's code and the completed experiment.
- Same sampler/evaluation: deterministic 40-step Heun, $\sigma_{\max}=80$, 100 generated
  samples, fixed latent schedule, `exclude_nn=True`, `aggregate='mean_of_ratios'`, and the
  locked coarse/mid1/mid2/fine bands.
  **Sampler lineage:** this notebook deliberately retains 80/40 because it directly extends the
  completed 25-arm SongUNet result; changing the sampler would confound dataset size. Separate
  SmallUNet experiment notebooks use $\sigma_{\max}=10$ with 1,000 steps.
- Because the raw ring ratio is intrinsically $n$-dependent, the main figures divide every
  band score by a **per-$n$ neutral baseline**: the score of the same 100 held-out real fields
  against each nested training set, with a fixed reference-draw seed. Raw scores are retained
  for continuity with the $n=2$ result.

No `src/` files are changed. Completed arms and partial checkpoints are resumed idempotently.

**Cluster execution** (one process):

```bash
python3 -m nbconvert --to notebook --execute --inplace \
  --ExecutePreprocessor.timeout=-1 --ExecutePreprocessor.kernel_name=python3 \
  notebooks/multiscale/songunet_covariance_tikhonov_dataset_size.ipynb
```

For parallel GPUs, set `ARM_SHARD=0..K-1` and `ARM_NSHARDS=K`, and give each process a
different nbconvert `--output` path (do not let several jobs write this notebook in place).
Shards suppress figure-file writes by default; rerun once with the default single-shard
settings after all artifacts exist to assemble the final figures without retraining.
If the completed $n=2$ artifacts are elsewhere, set `SOURCE_N2_RESULTS_DIR` to their
directory. Set `FIELD_SMOKE=1`
for a separate, tiny end-to-end run whose files cannot be mistaken for full results.


## Setup

In [ ]:
import os, sys, math, time, copy, json
from contextlib import contextmanager
import numpy as np
import torch
import matplotlib

# Headless-safe: picks Agg under SLURM (no $DISPLAY), leaves inline alone in Jupyter.
if not os.environ.get('DISPLAY') and not hasattr(sys, 'ps1'):
    matplotlib.use('Agg')
import matplotlib.pyplot as plt

# -- path setup --
repo_root = os.path.abspath(os.path.join(os.getcwd(), '..', '..'))
src_dir = os.path.join(repo_root, 'src')
if src_dir not in sys.path:
    sys.path.insert(0, src_dir)

from device_utils import resolve_device

DEVICE = resolve_device()           # cuda > mps > cpu
torch.backends.cudnn.benchmark = True

GRID = 128
N_TRAIN_VALUES = [2, 4, 8, 16, 32]

# Redirect heavy artifacts off a shared quota: export FIELD_RESULTS_DIR=$SCRATCH/...
results_dir = os.environ.get('FIELD_RESULTS_DIR', os.path.join(repo_root, 'results', 'data'))
fig_dir = os.environ.get('FIELD_FIG_DIR', os.path.join(repo_root, 'results', 'figures'))
os.makedirs(results_dir, exist_ok=True)
os.makedirs(fig_dir, exist_ok=True)

print(f'torch {torch.__version__}')
print(f'results -> {results_dir}')
print(f'figures -> {fig_dir}')
print(f'device: {DEVICE}')
if DEVICE.type == 'cuda':
    p = torch.cuda.get_device_properties(0)
    print(f'gpu: {p.name}, {p.total_memory / 1e9:.1f} GB, cc {p.major}.{p.minor}')
else:
    print('WARNING: full runs are sized for CUDA. MPS sampling is float32 rather than the '
          'EDM float64 reference; use FIELD_SMOKE=1 only for local pipeline checks.')


## The EDM network -- verbatim

Copied byte-for-byte from `baptista_config_matern_n2.ipynb`, which transcribes
`RectangleImages/training/networks.py` (byte-identical to
[NVlabs/edm](https://github.com/NVlabs/edm) apart from comments) minus the `@persistence`
decorators and the unused `DhariwalUNet` / `VPPrecond` / `VEPrecond` / `iDDPMPrecond`.

> Karras, Aittala, Aila & Laine, *Elucidating the Design Space of Diffusion-Based Generative
> Models*, NeurIPS 2022. Code (c) 2022 NVIDIA CORPORATION & AFFILIATES, released under
> CC BY-NC-SA 4.0.

In [ ]:
# ---------------------------------------------------------------------------------------------
# Transcribed verbatim from RectangleImages/training/networks.py in baptistar/DiffusionModelDynamics
# (byte-identical to NVlabs/edm training/networks.py apart from comments).
#
# Copyright (c) 2022, NVIDIA CORPORATION & AFFILIATES. All rights reserved.
# Licensed under CC BY-NC-SA 4.0 -- http://creativecommons.org/licenses/by-nc-sa/4.0/
# "Elucidating the Design Space of Diffusion-Based Generative Models", Karras et al., NeurIPS 2022.
#
# Changes: @persistence decorators and the torch_utils import removed (pickle plumbing only);
# DhariwalUNet / VPPrecond / VEPrecond / iDDPMPrecond omitted (unused by main.py).
# No computational change.
# ---------------------------------------------------------------------------------------------

from torch.nn.functional import silu

def weight_init(shape, mode, fan_in, fan_out):
    if mode == 'xavier_uniform': return np.sqrt(6 / (fan_in + fan_out)) * (torch.rand(*shape) * 2 - 1)
    if mode == 'xavier_normal':  return np.sqrt(2 / (fan_in + fan_out)) * torch.randn(*shape)
    if mode == 'kaiming_uniform': return np.sqrt(3 / fan_in) * (torch.rand(*shape) * 2 - 1)
    if mode == 'kaiming_normal':  return np.sqrt(1 / fan_in) * torch.randn(*shape)
    raise ValueError(f'Invalid init mode "{mode}"')

#----------------------------------------------------------------------------
# Fully-connected layer.

class Linear(torch.nn.Module):
    def __init__(self, in_features, out_features, bias=True, init_mode='kaiming_normal', init_weight=1, init_bias=0):
        super().__init__()
        self.in_features = in_features
        self.out_features = out_features
        init_kwargs = dict(mode=init_mode, fan_in=in_features, fan_out=out_features)
        self.weight = torch.nn.Parameter(weight_init([out_features, in_features], **init_kwargs) * init_weight)
        self.bias = torch.nn.Parameter(weight_init([out_features], **init_kwargs) * init_bias) if bias else None

    def forward(self, x):
        x = x @ self.weight.to(x.dtype).t()
        if self.bias is not None:
            x = x.add_(self.bias.to(x.dtype))
        return x

#----------------------------------------------------------------------------
# Convolutional layer with optional up/downsampling.

class Conv2d(torch.nn.Module):
    def __init__(self,
        in_channels, out_channels, kernel, bias=True, up=False, down=False,
        resample_filter=[1,1], fused_resample=False, init_mode='kaiming_normal', init_weight=1, init_bias=0,
    ):
        assert not (up and down)
        super().__init__()
        self.in_channels = in_channels
        self.out_channels = out_channels
        self.up = up
        self.down = down
        self.fused_resample = fused_resample
        init_kwargs = dict(mode=init_mode, fan_in=in_channels*kernel*kernel, fan_out=out_channels*kernel*kernel)
        self.weight = torch.nn.Parameter(weight_init([out_channels, in_channels, kernel, kernel], **init_kwargs) * init_weight) if kernel else None
        self.bias = torch.nn.Parameter(weight_init([out_channels], **init_kwargs) * init_bias) if kernel and bias else None
        f = torch.as_tensor(resample_filter, dtype=torch.float32)
        f = f.ger(f).unsqueeze(0).unsqueeze(1) / f.sum().square()
        self.register_buffer('resample_filter', f if up or down else None)

    def forward(self, x):
        w = self.weight.to(x.dtype) if self.weight is not None else None
        b = self.bias.to(x.dtype) if self.bias is not None else None
        f = self.resample_filter.to(x.dtype) if self.resample_filter is not None else None
        w_pad = w.shape[-1] // 2 if w is not None else 0
        f_pad = (f.shape[-1] - 1) // 2 if f is not None else 0

        if self.fused_resample and self.up and w is not None:
            x = torch.nn.functional.conv_transpose2d(x, f.mul(4).tile([self.in_channels, 1, 1, 1]), groups=self.in_channels, stride=2, padding=max(f_pad - w_pad, 0))
            x = torch.nn.functional.conv2d(x, w, padding=max(w_pad - f_pad, 0))
        elif self.fused_resample and self.down and w is not None:
            x = torch.nn.functional.conv2d(x, w, padding=w_pad+f_pad)
            x = torch.nn.functional.conv2d(x, f.tile([self.out_channels, 1, 1, 1]), groups=self.out_channels, stride=2)
        else:
            if self.up:
                x = torch.nn.functional.conv_transpose2d(x, f.mul(4).tile([self.in_channels, 1, 1, 1]), groups=self.in_channels, stride=2, padding=f_pad)
            if self.down:
                x = torch.nn.functional.conv2d(x, f.tile([self.in_channels, 1, 1, 1]), groups=self.in_channels, stride=2, padding=f_pad)
            if w is not None:
                x = torch.nn.functional.conv2d(x, w, padding=w_pad)
        if b is not None:
            x = x.add_(b.reshape(1, -1, 1, 1))
        return x

#----------------------------------------------------------------------------
# Group normalization.

class GroupNorm(torch.nn.Module):
    def __init__(self, num_channels, num_groups=32, min_channels_per_group=4, eps=1e-5):
        super().__init__()
        self.num_groups = min(num_groups, num_channels // min_channels_per_group)
        self.eps = eps
        self.weight = torch.nn.Parameter(torch.ones(num_channels))
        self.bias = torch.nn.Parameter(torch.zeros(num_channels))

    def forward(self, x):
        x = torch.nn.functional.group_norm(x, num_groups=self.num_groups, weight=self.weight.to(x.dtype), bias=self.bias.to(x.dtype), eps=self.eps)
        return x

#----------------------------------------------------------------------------
# Attention weight computation, i.e., softmax(Q^T * K).
# Performs all computation using FP32, but uses the original datatype for
# inputs/outputs/gradients to conserve memory.

class AttentionOp(torch.autograd.Function):
    @staticmethod
    def forward(ctx, q, k):
        w = torch.einsum('ncq,nck->nqk', q.to(torch.float32), (k / np.sqrt(k.shape[1])).to(torch.float32)).softmax(dim=2).to(q.dtype)
        ctx.save_for_backward(q, k, w)
        return w

    @staticmethod
    def backward(ctx, dw):
        q, k, w = ctx.saved_tensors
        db = torch._softmax_backward_data(grad_output=dw.to(torch.float32), output=w.to(torch.float32), dim=2, input_dtype=torch.float32)
        dq = torch.einsum('nck,nqk->ncq', k.to(torch.float32), db).to(q.dtype) / np.sqrt(k.shape[1])
        dk = torch.einsum('ncq,nqk->nck', q.to(torch.float32), db).to(k.dtype) / np.sqrt(k.shape[1])
        return dq, dk

#----------------------------------------------------------------------------
# Unified U-Net block with optional up/downsampling and self-attention.
# Represents the union of all features employed by the DDPM++, NCSN++, and
# ADM architectures.

class UNetBlock(torch.nn.Module):
    def __init__(self,
        in_channels, out_channels, emb_channels, up=False, down=False, attention=False,
        num_heads=None, channels_per_head=64, dropout=0, skip_scale=1, eps=1e-5,
        resample_filter=[1,1], resample_proj=False, adaptive_scale=True,
        init=dict(), init_zero=dict(init_weight=0), init_attn=None,
    ):
        super().__init__()
        self.in_channels = in_channels
        self.out_channels = out_channels
        self.emb_channels = emb_channels
        self.num_heads = 0 if not attention else num_heads if num_heads is not None else out_channels // channels_per_head
        self.dropout = dropout
        self.skip_scale = skip_scale
        self.adaptive_scale = adaptive_scale

        self.norm0 = GroupNorm(num_channels=in_channels, eps=eps)
        self.conv0 = Conv2d(in_channels=in_channels, out_channels=out_channels, kernel=3, up=up, down=down, resample_filter=resample_filter, **init)
        self.affine = Linear(in_features=emb_channels, out_features=out_channels*(2 if adaptive_scale else 1), **init)
        self.norm1 = GroupNorm(num_channels=out_channels, eps=eps)
        self.conv1 = Conv2d(in_channels=out_channels, out_channels=out_channels, kernel=3, **init_zero)

        self.skip = None
        if out_channels != in_channels or up or down:
            kernel = 1 if resample_proj or out_channels!= in_channels else 0
            self.skip = Conv2d(in_channels=in_channels, out_channels=out_channels, kernel=kernel, up=up, down=down, resample_filter=resample_filter, **init)

        if self.num_heads:
            self.norm2 = GroupNorm(num_channels=out_channels, eps=eps)
            self.qkv = Conv2d(in_channels=out_channels, out_channels=out_channels*3, kernel=1, **(init_attn if init_attn is not None else init))
            self.proj = Conv2d(in_channels=out_channels, out_channels=out_channels, kernel=1, **init_zero)

    def forward(self, x, emb):
        orig = x
        x = self.conv0(silu(self.norm0(x)))

        params = self.affine(emb).unsqueeze(2).unsqueeze(3).to(x.dtype)
        if self.adaptive_scale:
            scale, shift = params.chunk(chunks=2, dim=1)
            x = silu(torch.addcmul(shift, self.norm1(x), scale + 1))
        else:
            x = silu(self.norm1(x.add_(params)))

        x = self.conv1(torch.nn.functional.dropout(x, p=self.dropout, training=self.training))
        x = x.add_(self.skip(orig) if self.skip is not None else orig)
        x = x * self.skip_scale

        if self.num_heads:
            q, k, v = self.qkv(self.norm2(x)).reshape(x.shape[0] * self.num_heads, x.shape[1] // self.num_heads, 3, -1).unbind(2)
            w = AttentionOp.apply(q, k)
            a = torch.einsum('nqk,nck->ncq', w, v)
            x = self.proj(a.reshape(*x.shape)).add_(x)
            x = x * self.skip_scale
        return x

#----------------------------------------------------------------------------
# Timestep embedding used in the DDPM++ and ADM architectures.

class PositionalEmbedding(torch.nn.Module):
    def __init__(self, num_channels, max_positions=10000, endpoint=False):
        super().__init__()
        self.num_channels = num_channels
        self.max_positions = max_positions
        self.endpoint = endpoint

    def forward(self, x):
        freqs = torch.arange(start=0, end=self.num_channels//2, dtype=torch.float32, device=x.device)
        freqs = freqs / (self.num_channels // 2 - (1 if self.endpoint else 0))
        freqs = (1 / self.max_positions) ** freqs
        x = x.ger(freqs.to(x.dtype))
        x = torch.cat([x.cos(), x.sin()], dim=1)
        return x

#----------------------------------------------------------------------------
# Timestep embedding used in the NCSN++ architecture.

class FourierEmbedding(torch.nn.Module):
    def __init__(self, num_channels, scale=16):
        super().__init__()
        self.register_buffer('freqs', torch.randn(num_channels // 2) * scale)

    def forward(self, x):
        x = x.ger((2 * np.pi * self.freqs).to(x.dtype))
        x = torch.cat([x.cos(), x.sin()], dim=1)
        return x

#----------------------------------------------------------------------------
# Reimplementation of the DDPM++ and NCSN++ architectures from the paper
# "Score-Based Generative Modeling through Stochastic Differential
# Equations". Equivalent to the original implementation by Song et al.,
# available at https://github.com/yang-song/score_sde_pytorch

class SongUNet(torch.nn.Module):
    def __init__(self,
        img_resolution,                     # Image resolution at input/output.
        in_channels,                        # Number of color channels at input.
        out_channels,                       # Number of color channels at output.
        label_dim           = 0,            # Number of class labels, 0 = unconditional.
        augment_dim         = 0,            # Augmentation label dimensionality, 0 = no augmentation.

        model_channels      = 128,          # Base multiplier for the number of channels.
        channel_mult        = [1,2,2,2],    # Per-resolution multipliers for the number of channels.
        channel_mult_emb    = 4,            # Multiplier for the dimensionality of the embedding vector.
        num_blocks          = 4,            # Number of residual blocks per resolution.
        attn_resolutions    = [16],         # List of resolutions with self-attention.
        dropout             = 0.10,         # Dropout probability of intermediate activations.
        label_dropout       = 0,            # Dropout probability of class labels for classifier-free guidance.

        embedding_type      = 'positional', # Timestep embedding type: 'positional' for DDPM++, 'fourier' for NCSN++.
        channel_mult_noise  = 1,            # Timestep embedding size: 1 for DDPM++, 2 for NCSN++.
        encoder_type        = 'standard',   # Encoder architecture: 'standard' for DDPM++, 'residual' for NCSN++.
        decoder_type        = 'standard',   # Decoder architecture: 'standard' for both DDPM++ and NCSN++.
        resample_filter     = [1,1],        # Resampling filter: [1,1] for DDPM++, [1,3,3,1] for NCSN++.
    ):
        assert embedding_type in ['fourier', 'positional']
        assert encoder_type in ['standard', 'skip', 'residual']
        assert decoder_type in ['standard', 'skip']

        super().__init__()
        self.label_dropout = label_dropout
        emb_channels = model_channels * channel_mult_emb
        noise_channels = model_channels * channel_mult_noise
        init = dict(init_mode='xavier_uniform')
        init_zero = dict(init_mode='xavier_uniform', init_weight=1e-5)
        init_attn = dict(init_mode='xavier_uniform', init_weight=np.sqrt(0.2))
        block_kwargs = dict(
            emb_channels=emb_channels, num_heads=1, dropout=dropout, skip_scale=np.sqrt(0.5), eps=1e-6,
            resample_filter=resample_filter, resample_proj=True, adaptive_scale=False,
            init=init, init_zero=init_zero, init_attn=init_attn,
        )

        # Mapping.
        self.map_noise = PositionalEmbedding(num_channels=noise_channels, endpoint=True) if embedding_type == 'positional' else FourierEmbedding(num_channels=noise_channels)
        self.map_label = Linear(in_features=label_dim, out_features=noise_channels, **init) if label_dim else None
        self.map_augment = Linear(in_features=augment_dim, out_features=noise_channels, bias=False, **init) if augment_dim else None
        self.map_layer0 = Linear(in_features=noise_channels, out_features=emb_channels, **init)
        self.map_layer1 = Linear(in_features=emb_channels, out_features=emb_channels, **init)

        # Encoder.
        self.enc = torch.nn.ModuleDict()
        cout = in_channels
        caux = in_channels
        for level, mult in enumerate(channel_mult):
            res = img_resolution >> level
            if level == 0:
                cin = cout
                cout = model_channels
                self.enc[f'{res}x{res}_conv'] = Conv2d(in_channels=cin, out_channels=cout, kernel=3, **init)
            else:
                self.enc[f'{res}x{res}_down'] = UNetBlock(in_channels=cout, out_channels=cout, down=True, **block_kwargs)
                if encoder_type == 'skip':
                    self.enc[f'{res}x{res}_aux_down'] = Conv2d(in_channels=caux, out_channels=caux, kernel=0, down=True, resample_filter=resample_filter)
                    self.enc[f'{res}x{res}_aux_skip'] = Conv2d(in_channels=caux, out_channels=cout, kernel=1, **init)
                if encoder_type == 'residual':
                    self.enc[f'{res}x{res}_aux_residual'] = Conv2d(in_channels=caux, out_channels=cout, kernel=3, down=True, resample_filter=resample_filter, fused_resample=True, **init)
                    caux = cout
            for idx in range(num_blocks):
                cin = cout
                cout = model_channels * mult
                attn = (res in attn_resolutions)
                self.enc[f'{res}x{res}_block{idx}'] = UNetBlock(in_channels=cin, out_channels=cout, attention=attn, **block_kwargs)
        skips = [block.out_channels for name, block in self.enc.items() if 'aux' not in name]

        # Decoder.
        self.dec = torch.nn.ModuleDict()
        for level, mult in reversed(list(enumerate(channel_mult))):
            res = img_resolution >> level
            if level == len(channel_mult) - 1:
                self.dec[f'{res}x{res}_in0'] = UNetBlock(in_channels=cout, out_channels=cout, attention=True, **block_kwargs)
                self.dec[f'{res}x{res}_in1'] = UNetBlock(in_channels=cout, out_channels=cout, **block_kwargs)
            else:
                self.dec[f'{res}x{res}_up'] = UNetBlock(in_channels=cout, out_channels=cout, up=True, **block_kwargs)
            for idx in range(num_blocks + 1):
                cin = cout + skips.pop()
                cout = model_channels * mult
                attn = (idx == num_blocks and res in attn_resolutions)
                self.dec[f'{res}x{res}_block{idx}'] = UNetBlock(in_channels=cin, out_channels=cout, attention=attn, **block_kwargs)
            if decoder_type == 'skip' or level == 0:
                if decoder_type == 'skip' and level < len(channel_mult) - 1:
                    self.dec[f'{res}x{res}_aux_up'] = Conv2d(in_channels=out_channels, out_channels=out_channels, kernel=0, up=True, resample_filter=resample_filter)
                self.dec[f'{res}x{res}_aux_norm'] = GroupNorm(num_channels=cout, eps=1e-6)
                self.dec[f'{res}x{res}_aux_conv'] = Conv2d(in_channels=cout, out_channels=out_channels, kernel=3, **init_zero)

    def forward(self, x, noise_labels, class_labels, augment_labels=None):
        # Mapping.
        emb = self.map_noise(noise_labels)
        emb = emb.reshape(emb.shape[0], 2, -1).flip(1).reshape(*emb.shape) # swap sin/cos
        if self.map_label is not None:
            tmp = class_labels
            if self.training and self.label_dropout:
                tmp = tmp * (torch.rand([x.shape[0], 1], device=x.device) >= self.label_dropout).to(tmp.dtype)
            emb = emb + self.map_label(tmp * np.sqrt(self.map_label.in_features))
        if self.map_augment is not None and augment_labels is not None:
            emb = emb + self.map_augment(augment_labels)
        emb = silu(self.map_layer0(emb))
        emb = silu(self.map_layer1(emb))

        # Encoder.
        skips = []
        aux = x
        for name, block in self.enc.items():
            if 'aux_down' in name:
                aux = block(aux)
            elif 'aux_skip' in name:
                x = skips[-1] = x + block(aux)
            elif 'aux_residual' in name:
                x = skips[-1] = aux = (x + block(aux)) / np.sqrt(2)
            else:
                x = block(x, emb) if isinstance(block, UNetBlock) else block(x)
                skips.append(x)

        # Decoder.
        aux = None
        tmp = None
        for name, block in self.dec.items():
            if 'aux_up' in name:
                aux = block(aux)
            elif 'aux_norm' in name:
                tmp = block(x)
            elif 'aux_conv' in name:
                tmp = block(silu(tmp))
                aux = tmp if aux is None else tmp + aux
            else:
                if x.shape[1] != block.in_channels:
                    x = torch.cat([x, skips.pop()], dim=1)
                x = block(x, emb)
        return aux

In [ ]:
#----------------------------------------------------------------------------
class EDMPrecond(torch.nn.Module):
    def __init__(self,
        img_resolution,                     # Image resolution.
        img_channels,                       # Number of color channels.
        label_dim       = 0,                # Number of class labels, 0 = unconditional.
        use_fp16        = False,            # Execute the underlying model at FP16 precision?
        sigma_min       = 0,                # Minimum supported noise level.
        sigma_max       = float('inf'),     # Maximum supported noise level.
        sigma_data      = 0.5,              # Expected standard deviation of the training data.
        model_type      = 'DhariwalUNet',   # Class name of the underlying model.
        **model_kwargs,                     # Keyword arguments for the underlying model.
    ):
        super().__init__()
        self.img_resolution = img_resolution
        self.img_channels = img_channels
        self.label_dim = label_dim
        self.use_fp16 = use_fp16
        self.sigma_min = sigma_min
        self.sigma_max = sigma_max
        self.sigma_data = sigma_data
        self.model = globals()[model_type](img_resolution=img_resolution, in_channels=img_channels, out_channels=img_channels, label_dim=label_dim, **model_kwargs)

    def forward(self, x, sigma, class_labels=None, force_fp32=False, **model_kwargs):
        x = x.to(torch.float32)
        sigma = sigma.to(torch.float32).reshape(-1, 1, 1, 1)
        class_labels = None if self.label_dim == 0 else torch.zeros([1, self.label_dim], device=x.device) if class_labels is None else class_labels.to(torch.float32).reshape(-1, self.label_dim)
        dtype = torch.float16 if (self.use_fp16 and not force_fp32 and x.device.type == 'cuda') else torch.float32

        c_skip = self.sigma_data ** 2 / (sigma ** 2 + self.sigma_data ** 2) #1
        c_out = sigma * self.sigma_data / (sigma ** 2 + self.sigma_data ** 2).sqrt() #0
        c_in = 1 / (self.sigma_data ** 2 + sigma ** 2).sqrt()
        c_noise = sigma.log() / 4

        F_x = self.model((c_in * x).to(dtype), c_noise.flatten(), class_labels=class_labels, **model_kwargs)
        assert F_x.dtype == dtype
        D_x = c_skip * x + c_out * F_x.to(torch.float32)
        return D_x

    def round_sigma(self, sigma):
        return torch.as_tensor(sigma)

### Network config and verification

In [ ]:
# main.py:50-57, with img_resolution and attn_resolutions carrying the data change.
CHANNEL_MULT = [2, 2, 2]
ATTN_RES = [GRID >> (len(CHANNEL_MULT) - 1)]    # deepest level: 128>>2 = 32  (main.py: 64>>2 = 16)
assert ATTN_RES == [32], ATTN_RES

NET_KWARGS = dict(
    img_resolution   = GRID,        # main.py:55 has 64; our fields are 128
    img_channels     = 1,
    label_dim        = 0,
    use_fp16         = False,
    model_type       = 'SongUNet',
    embedding_type   = 'positional',
    encoder_type     = 'standard',
    decoder_type     = 'standard',
    channel_mult_noise = 1,
    resample_filter  = [1, 1],
    channel_mult     = CHANNEL_MULT,
    dropout          = 0.0,
    attn_resolutions = ATTN_RES,    # main.py leaves the SongUNet default [16]; see markdown above
)
# num_blocks=4 is a SongUNet default; main.py does not override it.

MODEL_CHANNELS = 16   # 880,097 params -- cheapest capacity that fully memorizes at n=2

def build_net(model_channels, device=None):
    net = EDMPrecond(model_channels=model_channels, **NET_KWARGS)
    return net if device is None else net.to(device)

def count_params(net):
    return sum(p.numel() for p in net.parameters())

PAPER_FIG18_PARAMS = {4: 57017, 8: 222705, 16: 880097,
                      32: 3498945, 64: 13952897, 128: 55725825}

n = count_params(build_net(MODEL_CHANNELS))
assert n == PAPER_FIG18_PARAMS[MODEL_CHANNELS], (
    f'model_channels={MODEL_CHANNELS}: got {n:,}, Figure 18 says {PAPER_FIG18_PARAMS[MODEL_CHANNELS]:,}')
print(f'model_channels={MODEL_CHANNELS}: {n:,} params (matches Figure 18)')

## The loss -- verbatim

`training/loss.py:65-81`. `main.py:63-65` constructs `EDMLoss` with no arguments, so `P_mean=-1.2`,
`P_std=1.2`, `sigma_data=0.5` are all class defaults.

In [ ]:
class EDMLoss:
    """training/loss.py, verbatim. Returns the per-element loss; the caller reduces it."""
    def __init__(self, P_mean=-1.2, P_std=1.2, sigma_data=0.5):
        self.P_mean = P_mean
        self.P_std = P_std
        self.sigma_data = sigma_data

    def __call__(self, net, images, labels=None, augment_pipe=None):
        rnd_normal = torch.randn([images.shape[0], 1, 1, 1], device=images.device)
        sigma = (rnd_normal * self.P_std + self.P_mean).exp()
        weight = (sigma ** 2 + self.sigma_data ** 2) / (sigma * self.sigma_data) ** 2
        y, augment_labels = augment_pipe(images) if augment_pipe is not None else (images, None)
        n = torch.randn_like(y) * sigma
        D_yn = net(y + n, sigma, labels, augment_labels=augment_labels)
        loss = weight * ((D_yn - y) ** 2)
        return loss

## The sampler -- verbatim

`generate.py:25-60`, EDM Algorithm 2; `main.py:134` leaves `S_churn=0`, so no noise is injected and
Algorithm 2 degenerates to deterministic 2nd-order Heun -- an ODE.

In [ ]:
# float64 everywhere, exactly as EDM -- except on MPS, which cannot allocate float64 at all.
SAMPLER_DTYPE = torch.float32 if DEVICE.type == 'mps' else torch.float64
if SAMPLER_DTYPE is torch.float32:
    print('WARNING: MPS cannot do float64; sampling in float32. EDM (and a CUDA run) uses '
          'float64 -- do not report numbers from an MPS run.')


def edm_sampler(
    net, latents, class_labels=None, randn_like=torch.randn_like,
    num_steps=18, sigma_min=0.002, sigma_max=80, rho=7,
    S_churn=0, S_min=0, S_max=float('inf'), S_noise=1,
):
    """generate.py, verbatim (EDM Algorithm 2)."""
    # Adjust noise levels based on what's supported by the network.
    sigma_min = max(sigma_min, net.sigma_min)
    sigma_max = min(sigma_max, net.sigma_max)

    # Time step discretization.
    step_indices = torch.arange(num_steps, dtype=SAMPLER_DTYPE, device=latents.device)  # EDM: torch.float64
    t_steps = (sigma_max ** (1 / rho) + step_indices / (num_steps - 1) * (sigma_min ** (1 / rho) - sigma_max ** (1 / rho))) ** rho
    t_steps = torch.cat([net.round_sigma(t_steps), torch.zeros_like(t_steps[:1])]) # t_N = 0

    # Main sampling loop.
    x_next = latents.to(SAMPLER_DTYPE) * t_steps[0]  # EDM: torch.float64
    for i, (t_cur, t_next) in enumerate(zip(t_steps[:-1], t_steps[1:])): # 0, ..., N-1
        x_cur = x_next

        # Increase noise temporarily.
        gamma = min(S_churn / num_steps, np.sqrt(2) - 1) if S_min <= t_cur <= S_max else 0
        t_hat = net.round_sigma(t_cur + gamma * t_cur)
        x_hat = x_cur + (t_hat ** 2 - t_cur ** 2).sqrt() * S_noise * randn_like(x_cur)

        # Euler step.
        denoised = net(x_hat, t_hat, class_labels).to(SAMPLER_DTYPE)  # EDM: torch.float64
        d_cur = (x_hat - denoised) / t_hat
        x_next = x_hat + (t_next - t_hat) * d_cur

        # Apply 2nd order correction.
        if i < num_steps - 1:
            denoised = net(x_next, t_next, class_labels).to(SAMPLER_DTYPE)  # EDM: torch.float64
            d_prime = (x_next - denoised) / t_next
            x_next = x_hat + (t_next - t_hat) * (0.5 * d_cur + 0.5 * d_prime)

    return x_next

## Configuration

## Configuration

The only training-axis change from the completed notebook is replacing 50,000 epochs with the
equivalent fixed budget of 100,000 optimizer updates. At $n=2$ these are identical.


In [ ]:
SMOKE = os.environ.get('FIELD_SMOKE', '0').strip().lower() not in ('0', '', 'false', 'no')

CFG = dict(
    # -- exact completed-run training configuration -------------------------------------------
    max_updates        = 100_000,   # n=2 source: 50,000 epochs x 2 batch-1 updates
    batch_size         = 1,
    lr                 = 10e-4,
    betas              = (0.9, 0.999),
    eps                = 1e-8,
    lr_rampup_kimg     = 10_000,
    ema_halflife_kimg  = 500,
    ema_rampup_ratio   = 0.05,
    P_mean             = -1.2,
    P_std              = 1.2,
    sigma_data         = 0.5,

    # -- exact completed-run sampler ----------------------------------------------------------
    num_steps          = 40,
    sigma_min          = 0.002,
    sigma_max          = 80.0,
    rho                = 7,
    S_churn            = 0.0,
    S_min              = 0.0,
    S_max              = float('inf'),
    S_noise            = 1.0,

    # -- matched evaluation ------------------------------------------------------------------
    n_eval_samples     = 100,
    eval_every_updates = 2_000,     # 50 checkpoints, exactly the n=2 source cadence
    rel_threshold      = 0.3,
    n_rand_ref         = 32,
    metric_ref_seed    = 31_415,
    neutral_ref_seed   = 27_182,
    neutral_start      = 32,        # same held-out fields for every n
    n_neutral          = 100,

    # -- mechanics ---------------------------------------------------------------------------
    seed               = 0,
    latent_seed        = 42,
    eval_batch         = 25,
    save_resume_state  = True,
    n_sample_grid      = 16,
)

if SMOKE:
    N_TRAIN_VALUES = [2, 4]
    CFG.update(max_updates=16, eval_every_updates=8, n_eval_samples=4,
               n_neutral=8, num_steps=4, eval_batch=4,
               save_resume_state=False, n_sample_grid=4)

assert CFG['batch_size'] == 1, 'this notebook reproduces the completed batch-1 experiment'
assert all(CFG['max_updates'] % n == 0 for n in N_TRAIN_VALUES)
assert CFG['max_updates'] % CFG['eval_every_updates'] == 0

RUN_TAG = 'smoke' if SMOKE else 'full'
RESULT_DIR = os.path.join(results_dir, f'songunet_cov_tikhonov_dataset_size_{RUN_TAG}')
os.makedirs(RESULT_DIR, exist_ok=True)
STATE_DIR = RESULT_DIR

SOURCE_N2_DIR = os.environ.get(
    'SOURCE_N2_RESULTS_DIR',
    os.path.join(repo_root, 'results', 'data', 'songunet_cov_tikhonov'),
)
REUSE_N2 = (not SMOKE) and os.environ.get('REUSE_N2', '1').strip().lower() not in ('0', 'false', 'no')

def arm_result_path(n_train, variant, c_val):
    return os.path.join(RESULT_DIR, f'arm_n{n_train}_{variant}_c{c_val:g}_result.pt')

def arm_state_path(n_train, variant, c_val):
    return os.path.join(STATE_DIR, f'arm_n{n_train}_{variant}_c{c_val:g}.pt')

def source_n2_path(variant, c_val):
    return os.path.join(SOURCE_N2_DIR, f'arm_{variant}_c{c_val:g}_result.pt')

print(f"{'SMOKE RUN' if SMOKE else 'FULL RUN'}")
print(f'  n_train       : {N_TRAIN_VALUES}')
print(f"  updates / arm : {CFG['max_updates']:,}")
print(f"  eval cadence  : {CFG['eval_every_updates']:,} updates")
print(f"  sigma_max     : {CFG['sigma_max']}")
print(f"  sigma_data    : {CFG['sigma_data']}")
print(f'  reuse n=2     : {REUSE_N2} from {SOURCE_N2_DIR}')
print(f'  results dir   : {RESULT_DIR}')


## Training data

This is the exact 200-field construction used by the completed 25-arm notebook. The scale is
computed once from its first two fields, then applied unchanged to the entire pool. The nested
training sets therefore change only how many leading fields are included.


In [ ]:
from multiband_data_utils import generate_multiband_dataset_postmask, make_knrm_grid

components = [
    {'name': 'coarse', 'length_scale': 2.0,  's': 2.0, 'sigma_sq': 1.0, 'band': (0.5, 4.0)},
    {'name': 'mid1',   'length_scale': 6.0,  's': 2.0, 'sigma_sq': 1.0, 'band': (4.0, 10.0)},
    {'name': 'mid2',   'length_scale': 12.0, 's': 2.0, 'sigma_sq': 1.0, 'band': (10.0, 18.0)},
    {'name': 'fine',   'length_scale': 24.0, 's': 2.0, 'sigma_sq': 1.0, 'band': (18.0, 32.0)},
]
weights = [1.0, 0.8, 0.8, 1.2]

_pool = generate_multiband_dataset_postmask(
    num_samples=200, grid_size=GRID, components=components,
    weights=weights, seed=42, normalize=True,
)
normalization_std = _pool['normalization']['std']
_base_all = _pool['combined'].reshape(200, 1, GRID, GRID).clone().float()

def _pair_dist(x):
    f = x.reshape(x.shape[0], -1)
    return torch.cdist(f, f)[0, 1].item()

RECT_D_MINUS = math.sqrt(340.0)
D_UNIT = _pair_dist(_base_all[:2])
SCALE = RECT_D_MINUS / D_UNIT
data_all = _base_all * SCALE
D_MINUS = _pair_dist(data_all[:2])
assert abs(D_MINUS - RECT_D_MINUS) < 1e-3

# Strong local provenance guard: the completed n=2 fields must match bit-for-bit.
_ref_gate_path = source_n2_path('gate', 0.0)
if os.path.exists(_ref_gate_path):
    _ref_gate = torch.load(_ref_gate_path, map_location='cpu', weights_only=False)
    # FFT kernels can drift by a few float32 ulps across Torch/platform versions.
    # Verify scientific identity, then anchor the first two tensors to the completed artifact
    # so every larger nested set contains the exact fields used in the 25-arm experiment.
    _data_delta = (data_all[:2] - _ref_gate['data']).abs().max().item()
    assert torch.allclose(data_all[:2], _ref_gate['data'], rtol=1e-6, atol=1e-7), (
        f'n=2 regenerated fields materially differ; max_abs={_data_delta:.3e}')
    data_all[:2] = _ref_gate['data']
    assert torch.equal(data_all[:2], _ref_gate['data'])
    D_MINUS = _pair_dist(data_all[:2])
    assert abs(D_MINUS - RECT_D_MINUS) < 1e-3
    print(f'PASSED: regenerated n=2 fields agree within {_data_delta:.3e}; anchored to artifact')
else:
    print(f'n=2 provenance artifact not found; numerical construction guards still active: {_ref_gate_path}')

neutral_end = CFG['neutral_start'] + CFG['n_neutral']
assert neutral_end <= data_all.shape[0]
neutral_fields = data_all[CFG['neutral_start']:neutral_end].squeeze(1).clone()

print(f'pool={tuple(data_all.shape)}, nested n={N_TRAIN_VALUES}')
print(f'D_UNIT={D_UNIT:.4f}, SCALE={SCALE:.8f}, D_minus={D_MINUS:.4f}')
print(f'rescaled pool std={data_all.std():.5f}, normalization_std={normalization_std:.6f}')
print(f'neutral fields: pool[{CFG["neutral_start"]}:{neutral_end}] (shared across n)')


## Fixed analytic covariance spectrum

The shape below is the closed-form spectrum of the known multiband Matérn generator after the
same normalization and rescaling as the data. The final budget normalization makes its global
scale irrelevant and matches the covariance-penalty budget to the isotropic pixel-sum penalty.
Null modes retain the completed experiment's pseudo-inverse semantics.


In [ ]:
knrm = make_knrm_grid(GRID).double()
ring = knrm.round().long()

lam_pop = torch.zeros(GRID, GRID, dtype=torch.float64)
for comp, w in zip(components, weights):
    S = comp['sigma_sq'] * (knrm**2 + comp['length_scale']**2) ** (-comp['s'])
    S[0, 0] = 0.0
    k_lo, k_hi = comp['band']
    S = S * ((knrm >= k_lo) & (knrm < k_hi)).double()
    lam_pop += (w**2) * S
lam_pop = lam_pop / (normalization_std**2) * (SCALE**2)

NULL_REL_THRESHOLD = 1e-4

def build_inv_lam(lam_2d):
    positive = lam_2d > 0
    active = positive & (lam_2d > NULL_REL_THRESHOLD * lam_2d[positive].mean())
    inv = torch.where(active, 1.0 / lam_2d.clamp_min(1e-30), torch.zeros_like(lam_2d))
    return inv, active

inv_lam_pop, active_pop = build_inv_lam(lam_pop)
M = GRID * GRID
inv_lam_pop = inv_lam_pop * (M * M / inv_lam_pop.sum())
assert torch.isfinite(inv_lam_pop).all()
assert abs(inv_lam_pop.sum().item() - M*M) < 1e-5 * M*M

# Exact guard against the analytic spectrum stored by the completed run.
if os.path.exists(_ref_gate_path):
    assert torch.equal(lam_pop, _ref_gate['lam_pop']), 'analytic lambda differs from completed run'
    print('PASSED: analytic lambda torch.equal the completed 25-arm run')

print(f'analytic spectrum active modes: {active_pop.sum().item()}/{GRID*GRID}')
print(f'budget-normalized sum(inv_lambda)={inv_lam_pop.sum().item():.1f} (= M^2)')


## Tikhonov penalties

Copied unchanged from the completed notebook. Both penalties use a pixel/mode sum and share the
EDM loss weight, sigma, noise, and denoiser forward pass with the data term.


In [ ]:
# ---------------------------------------------------------------------------
# Tikhonov penalty (notebook-local)
# ---------------------------------------------------------------------------

def tikhonov_penalty_cov(D_theta, x_noisy, sigma, c, inv_lam_2d):
    """Covariance-weighted Tikhonov penalty on the implied score.

    D_theta: denoiser output (B, 1, N, N)
    x_noisy: noisy input (B, 1, N, N)
    sigma: noise level (B,) or scalar
    c: regularization constant
    inv_lam_2d: 1/lambda(k), shape (N, N), with null modes zeroed

    Score = (D_theta - x_noisy) / sigma^2  (EDM preconditioning)
    Penalty per sample = c/sigma^2 * sum_k |score_hat_k|^2 / lambda(k)
           = c/sigma^4 * sum_k |FFT(D_theta - x_noisy)_k|^2 / lambda(k)

    The 'forward' FFT norm gives coefficients with the standard 1/N^2 scaling.
    We use SUM over modes (not mean) to match EDMLoss's sum-over-pixels reduction.

    Returns: (B,) per-sample penalty.
    """
    s2 = sigma.reshape(-1, 1, 1, 1) ** 2
    diff = D_theta - x_noisy  # (B, 1, N, N)
    diff_hat = torch.fft.fft2(diff.squeeze(1), norm='forward')  # (B, N, N)
    # Per-mode weighted: |diff_hat_k|^2 * (1/lambda(k))
    per_mode = diff_hat.abs() ** 2 * inv_lam_2d.unsqueeze(0).to(diff_hat.device)
    # Sum over spatial modes (matching sum-over-pixels in the data term)
    penalty_per_sample = per_mode.sum(dim=(-2, -1))  # (B,)
    # Divide by sigma^4 (score = diff/sigma^2, so |score|^2 ~ diff^2/sigma^4),
    # then multiply by c. But we want c/sigma^2 * sum|score_hat|^2 / lambda
    # = c/sigma^2 * sum|diff_hat/sigma^2|^2 / lambda = c/sigma^4 * sum|diff_hat|^2/lambda
    # Actually, let's rewrite:
    # The EDM loss weight already contains weight(sigma), and in Baptista's formulation
    # the Tikhonov penalty carries the same weight lambda(sigma) as the data term.
    # So the penalty here should be: c * ||D - x||^2_{inv_lam} / sigma^2
    # (analogous to src/edm.py:tikhonov_penalty which uses c * ((D-x)^2 / sigma^2).mean())
    # With sum reduction: c * sum_k |FFT(D-x)_k|^2 / (sigma^2 * lambda(k))
    return (c / s2.squeeze()) * penalty_per_sample


def tikhonov_penalty_iso(D_theta, x_noisy, sigma, c):
    """Isotropic Tikhonov penalty: c/sigma^2 * sum_pixels (D_theta - x_noisy)^2.

    Returns: (B,) per-sample penalty.
    """
    s2 = sigma.reshape(-1, 1, 1, 1) ** 2
    diff_sq = (D_theta - x_noisy) ** 2  # (B, 1, N, N)
    # Sum over all pixels (matching EDMLoss reduction)
    return c * diff_sq.sum(dim=(1, 2, 3)) / s2.squeeze()


print('Tikhonov penalty functions defined (notebook-local, sum reduction)')

## Arms

The `c` grid is frozen from the completed experiment; it is not recalibrated by dataset size.


In [ ]:
C_VALUES = [3e-3, 1e-2, 3e-2, 0.1]
if SMOKE:
    C_VALUES = [1e-2]

ARMS = []
for n_train in N_TRAIN_VALUES:
    ARMS.append(dict(n_train=n_train, variant='gate', c=0.0, inv_lam=None,
                     label=f'n={n_train} gate (c=0)'))
    for c in C_VALUES:
        ARMS.append(dict(n_train=n_train, variant='isotropic', c=c, inv_lam=None,
                         label=f'n={n_train} iso c={c:g}'))
    for c in C_VALUES:
        ARMS.append(dict(n_train=n_train, variant='cov_population', c=c,
                         inv_lam=inv_lam_pop.float().to(DEVICE),
                         label=f'n={n_train} analytic-cov c={c:g}'))

print(f'{len(ARMS)} conceptual arms; {sum(a["n_train"] > 2 for a in ARMS)} new larger-n arms')
for arm in ARMS:
    print(f'  {arm["label"]}')


## Metrics and per-$n$ neutral baselines

The reference-draw RNG is reset and restored around each metric call. Thus arms at a common
$(n,\mathrm{update})$ use identical random references without perturbing the training RNG.


In [ ]:
from memorization_metrics import RingMetricContext

bands = {c['name']: c['band'] for c in components}
ctx = RingMetricContext(GRID, bands, device=DEVICE)

@contextmanager
def preserved_seed(seed):
    cpu_state = torch.get_rng_state()
    cuda_state = torch.cuda.get_rng_state_all() if torch.cuda.is_available() else None
    mps_state = (torch.mps.get_rng_state()
                 if torch.backends.mps.is_available() and torch.backends.mps.is_built() else None)
    try:
        torch.manual_seed(int(seed))
        yield
    finally:
        torch.set_rng_state(cpu_state)
        if cuda_state is not None:
            torch.cuda.set_rng_state_all(cuda_state)
        if mps_state is not None:
            torch.mps.set_rng_state(mps_state)

@torch.no_grad()
def field_stats(x_gen, x_train, rel_threshold=0.3):
    g = x_gen.detach().float().cpu().reshape(x_gen.shape[0], -1)
    t = x_train.detach().float().cpu().reshape(x_train.shape[0], -1)
    d2 = torch.cdist(g, t) ** 2
    l2_min = d2.min(dim=1).values
    nn_idx = d2.argmin(dim=1)
    nn_rel = l2_min.clamp_min(0).sqrt() / t.norm(dim=1).mean()
    return {
        'l2_max': l2_min.max().item(), 'l2_mean': l2_min.mean().item(),
        'l2_min': l2_min.min().item(), 'nn_rel_median': nn_rel.median().item(),
        'nn_rel_min': nn_rel.min().item(),
        'fraction': (nn_rel < rel_threshold).float().mean().item(),
        'nn_index': nn_idx,
    }

def advance_source_metric_rng(idx_nn, n_train, n_ref):
    '''Consume exactly the random integers used by the source notebook's metric.

    The completed notebook let metric reference draws advance the device RNG between
    training checkpoints. We compute reported metrics under a fixed/restored seed, then
    reproduce that RNG advancement so the training stochastic process remains matched.
    '''
    idx_nn = idx_nn.to(DEVICE)
    for _ in range(n_ref):
        rand_idx = torch.randint(0, n_train, (idx_nn.shape[0],), device=DEVICE)
        if n_train > 1:
            clash = rand_idx == idx_nn
            if clash.any():
                shift = torch.randint(1, n_train, (int(clash.sum()),), device=DEVICE)
                rand_idx[clash] = (idx_nn[clash] + shift) % n_train

@torch.no_grad()
def ring_metric_eval(x_gen_2d, x_train_2d, metric_seed, advance_training_rng=False):
    with preserved_seed(metric_seed):
        m = ctx.evaluate(
            x_gen_2d.to(DEVICE), x_train_2d.to(DEVICE),
            n_rand_ref=CFG['n_rand_ref'], exclude_nn=True,
            aggregate='mean_of_ratios',
        )
    if advance_training_rng:
        advance_source_metric_rng(m['idx_nn'], x_train_2d.shape[0], CFG['n_rand_ref'])
    out = {'mean_ratio': m['mean_ratio'].cpu()}
    for name in bands:
        out[f'{name}_score'] = m[f'{name}_score'].mean().item()
    return out

neutral = {}
for n_train in N_TRAIN_VALUES:
    xtr = data_all[:n_train].squeeze(1)
    neutral[n_train] = ring_metric_eval(
        neutral_fields, xtr,
        metric_seed=CFG['neutral_ref_seed'] + n_train,
    )

print('Per-n neutral baselines (same held-out fields; raw mean-of-ratios):')
print(f'{"n":>4s} ' + ' '.join(f'{b:>9s}' for b in bands))
for n_train in N_TRAIN_VALUES:
    print(f'{n_train:>4d} ' + ' '.join(
        f'{neutral[n_train][f"{b}_score"]:>9.4f}' for b in bands))


## Closed-form sampler reachability checks

These are positive controls, not training arms. The exact empirical-Bayes denoiser is sampled
with the same deterministic Heun configuration at each dataset size.


In [ ]:
class GMMDenoiser(torch.nn.Module):
    '''Exact empirical-Bayes denoiser for the N-point empirical measure = full memorization.'''
    sigma_min = 0.0
    sigma_max = float('inf')

    def __init__(self, y):
        super().__init__()
        self.register_buffer('y', y.reshape(y.shape[0], -1))

    def round_sigma(self, sigma):
        return torch.as_tensor(sigma)

    def forward(self, x, sigma, class_labels=None):
        shp = x.shape
        xf = x.reshape(shp[0], -1).to(self.y.dtype)
        d2 = torch.cdist(xf, self.y) ** 2
        s = torch.as_tensor(sigma, dtype=self.y.dtype, device=xf.device).reshape(-1, 1)
        w = torch.softmax(-d2 / (2 * s ** 2), dim=1)
        return (w @ self.y).reshape(shp)


@torch.no_grad()
def run_gate(data_tensor, cfg):
    '''Push the closed-form memorizing denoiser through this sampler.'''
    g = torch.Generator().manual_seed(cfg['latent_seed'])
    lat = torch.randn(cfg['n_sample_grid'], 1, GRID, GRID, generator=g).to(DEVICE)
    gmm = GMMDenoiser(data_tensor.to(DEVICE).to(SAMPLER_DTYPE))
    xg = edm_sampler(gmm, lat, num_steps=cfg['num_steps'],
                     sigma_min=cfg['sigma_min'], sigma_max=cfg['sigma_max'], rho=cfg['rho'],
                     S_churn=cfg['S_churn'], S_min=cfg['S_min'], S_max=cfg['S_max'],
                     S_noise=cfg['S_noise'])
    st = field_stats(xg.float().cpu(), data_tensor, rel_threshold=cfg['rel_threshold'])
    st.pop('nn_index')
    st['passed'] = st['nn_rel_median'] < 0.05
    return st


GMM_GATES = {}
for n_train in N_TRAIN_VALUES:
    GMM_GATES[n_train] = run_gate(data_all[:n_train], CFG)
    g = GMM_GATES[n_train]
    print(f'n={n_train:>2}: nn_rel median={g["nn_rel_median"]:.5f}, '
          f'fraction={g["fraction"]:.2f}, passed={g["passed"]}')
assert all(g['passed'] for g in GMM_GATES.values()), 'sampler reachability gate failed'


## Training and evaluation

The network, optimizer, learning-rate ramp, EMA, loss reduction, and batch-1 shuffled loader are
unchanged. Evaluation occurs at exact multiples of 2,000 optimizer updates. For $n=32$ some
checkpoints fall halfway through an epoch; the current permutation and offset are stored so a
resumed run is exact. At $n=2$ this reproduces the completed source cadence.


In [ ]:
@torch.no_grad()
def generate_samples(ema_net, n_samples, cfg, device, seed):
    ema_net.eval()
    out, remaining, chunk_i = [], n_samples, 0
    while remaining > 0:
        b = min(cfg['eval_batch'], remaining)
        g = torch.Generator().manual_seed(seed + 1000 * chunk_i)
        latents = torch.randn(b, 1, GRID, GRID, generator=g).to(device)
        x = edm_sampler(
            ema_net, latents, num_steps=cfg['num_steps'],
            sigma_min=cfg['sigma_min'], sigma_max=cfg['sigma_max'], rho=cfg['rho'],
            S_churn=cfg['S_churn'], S_min=cfg['S_min'], S_max=cfg['S_max'],
            S_noise=cfg['S_noise'],
        )
        out.append(x.float().cpu())
        remaining -= b
        chunk_i += 1
    return torch.cat(out, dim=0)


def _materialize_epoch_order(n_train):
    '''Materialize the exact batch-1 DataLoader permutation for resumable mid-epoch saves.'''
    index_loader = torch.utils.data.DataLoader(
        torch.utils.data.TensorDataset(torch.arange(n_train)),
        batch_size=1, shuffle=True,
    )
    order = [int(idx.item()) for (idx,) in index_loader]
    assert sorted(order) == list(range(n_train))
    return order


def run_arm(arm, cfg, device, resume=True, log=print):
    n_train, variant, c_val = arm['n_train'], arm['variant'], arm['c']
    inv_lam_device = arm['inv_lam']
    state_path = arm_state_path(n_train, variant, c_val)

    torch.manual_seed(cfg['seed'])
    net = build_net(MODEL_CHANNELS, device)
    net.train().requires_grad_(True)
    ema = copy.deepcopy(net).eval().requires_grad_(False)
    optimizer = torch.optim.Adam(net.parameters(), lr=cfg['lr'],
                                 betas=list(cfg['betas']), eps=cfg['eps'])

    cur_nimg, opt_step, epoch_idx = 1, 0, 0
    batch_pos, epoch_order = 0, None
    epoch_loss_sum, epoch_count = 0.0, 0
    window_loss_sum, window_count = 0.0, 0
    eval_log, loss_hist = [], []
    if resume and os.path.exists(state_path):
        st = torch.load(state_path, map_location='cpu', weights_only=False)
        assert st.get('state_format') == 2, 'checkpoint predates exact-update resume format'
        assert st['n_train'] == n_train and st['max_updates'] == cfg['max_updates']
        assert st['eval_every_updates'] == cfg['eval_every_updates']
        net.load_state_dict(st['net']); ema.load_state_dict(st['ema'])
        optimizer.load_state_dict(st['opt'])
        cur_nimg, opt_step, epoch_idx = st['cur_nimg'], st['opt_step'], st['epoch_idx']
        batch_pos, epoch_order = st['batch_pos'], st['epoch_order']
        epoch_loss_sum, epoch_count = st['epoch_loss_sum'], st['epoch_count']
        window_loss_sum, window_count = st['window_loss_sum'], st['window_count']
        eval_log, loss_hist = st['eval_log'], st['loss_hist']
        torch.set_rng_state(st['rng_cpu'])
        if st.get('rng_cuda') is not None and torch.cuda.is_available():
            torch.cuda.set_rng_state_all(st['rng_cuda'])
        if st.get('rng_mps') is not None and torch.backends.mps.is_available():
            torch.mps.set_rng_state(st['rng_mps'])
        log(f'  resumed from update {opt_step:,} '
            f'(epoch {epoch_idx:,}, offset {batch_pos}/{n_train})')

    x_train_cpu = data_all[:n_train]
    x_train_2d = x_train_cpu.squeeze(1)
    start_step = opt_step
    t_start = time.time()

    while opt_step < cfg['max_updates']:
        if epoch_order is None:
            assert batch_pos == 0
            epoch_order = _materialize_epoch_order(n_train)
        assert 0 <= batch_pos < n_train
        sample_idx = epoch_order[batch_pos]
        x = x_train_cpu[sample_idx:sample_idx + 1].to(device)

        net.train()
        optimizer.zero_grad(set_to_none=True)
        rnd_normal = torch.randn([x.shape[0], 1, 1, 1], device=x.device)
        sigma = (rnd_normal * cfg['P_std'] + cfg['P_mean']).exp()
        weight = (sigma**2 + cfg['sigma_data']**2) / (sigma * cfg['sigma_data'])**2
        noise = torch.randn_like(x) * sigma
        x_noisy = x + noise
        D_theta = net(x_noisy, sigma)

        data_loss = weight * ((D_theta - x)**2)
        loss = data_loss.sum() / x.size(0)
        if c_val > 0:
            if inv_lam_device is not None:
                pen = tikhonov_penalty_cov(
                    D_theta, x_noisy, sigma.squeeze(), c_val, inv_lam_device)
            else:
                pen = tikhonov_penalty_iso(D_theta, x_noisy, sigma.squeeze(), c_val)
            loss = loss + (weight.squeeze() * pen).sum() / x.size(0)

        loss.backward()
        for group in optimizer.param_groups:
            group['lr'] = cfg['lr'] * min(
                cur_nimg / max(cfg['lr_rampup_kimg'] * 1000, 1e-8), 1)
        for param in net.parameters():
            if param.grad is not None:
                torch.nan_to_num(param.grad, nan=0, posinf=1e5, neginf=-1e5, out=param.grad)
        optimizer.step()

        ema_halflife_nimg = cfg['ema_halflife_kimg'] * 1000
        if cfg['ema_rampup_ratio'] is not None:
            ema_halflife_nimg = min(ema_halflife_nimg, cur_nimg * cfg['ema_rampup_ratio'])
        ema_beta = 0.5 ** (x.size(0) / max(ema_halflife_nimg, 1e-8))
        for p_ema, p_net in zip(ema.parameters(), net.parameters()):
            p_ema.copy_(p_net.detach().lerp(p_ema, ema_beta))

        loss_value = loss.item()
        cur_nimg += x.size(0)
        opt_step += 1
        batch_pos += 1
        epoch_loss_sum += loss_value
        epoch_count += x.size(0)
        window_loss_sum += loss_value
        window_count += x.size(0)

        if batch_pos == n_train:
            loss_hist.append(epoch_loss_sum / epoch_count)
            epoch_idx += 1
            batch_pos, epoch_order = 0, None
            epoch_loss_sum, epoch_count = 0.0, 0

        if opt_step % cfg['eval_every_updates'] == 0:
            n_done = opt_step // cfg['eval_every_updates']
            x_gen = generate_samples(
                ema, cfg['n_eval_samples'], cfg, device,
                seed=cfg['latent_seed'] + n_done,
            )
            st = field_stats(x_gen, x_train_cpu, rel_threshold=cfg['rel_threshold'])
            metric_seed = cfg['metric_ref_seed'] + n_train * 1_000_000 + opt_step
            ring_m = ring_metric_eval(
                x_gen.squeeze(1), x_train_2d, metric_seed, advance_training_rng=True)
            neu = neutral[n_train]

            row = dict(
                epoch=epoch_idx, examples_into_epoch=batch_pos,
                epoch_equivalent=opt_step / n_train,
                opt_step=opt_step, cur_nimg=cur_nimg,
                lr=optimizer.param_groups[0]['lr'],
                loss=window_loss_sum / max(window_count, 1),
                l2_max=st['l2_max'], l2_mean=st['l2_mean'], l2_min=st['l2_min'],
                nn_rel_median=st['nn_rel_median'], nn_rel_min=st['nn_rel_min'],
                fraction=st['fraction'], metric_ref_seed=metric_seed,
                mean_ratio=ring_m['mean_ratio'],
                neutral_mean_ratio=neu['mean_ratio'],
                samples=x_gen[:cfg['n_sample_grid']].clone(),
            )
            for band in bands:
                raw = ring_m[f'{band}_score']
                base = neu[f'{band}_score']
                row[f'{band}_score'] = raw
                row[f'{band}_neutral'] = base
                row[f'{band}_over_neutral'] = raw / base
            eval_log.append(row)
            window_loss_sum, window_count = 0.0, 0

            elapsed = time.time() - t_start
            done_this_session = max(opt_step - start_step, 1)
            eta = elapsed / done_this_session * (cfg['max_updates'] - opt_step)
            log(f'  update {opt_step:>7,} | epoch-equivalent {row["epoch_equivalent"]:>9.1f} | '
                f'lr {row["lr"]:.2e} | frac {st["fraction"]:.2f} | '
                f'coarse/base {row["coarse_over_neutral"]:.4f} '
                f'fine/base {row["fine_over_neutral"]:.4f} | '
                f'{elapsed/60:.1f}m elapsed, ~{eta/60:.0f}m left')

            if cfg['save_resume_state']:
                torch.save(dict(
                    state_format=2, net=net.state_dict(), ema=ema.state_dict(),
                    opt=optimizer.state_dict(), cur_nimg=cur_nimg, opt_step=opt_step,
                    epoch_idx=epoch_idx, batch_pos=batch_pos, epoch_order=epoch_order,
                    epoch_loss_sum=epoch_loss_sum, epoch_count=epoch_count,
                    window_loss_sum=window_loss_sum, window_count=window_count,
                    n_train=n_train, max_updates=cfg['max_updates'],
                    eval_every_updates=cfg['eval_every_updates'],
                    eval_log=eval_log, loss_hist=loss_hist,
                    rng_cpu=torch.get_rng_state(),
                    rng_cuda=(torch.cuda.get_rng_state_all() if torch.cuda.is_available() else None),
                    rng_mps=(torch.mps.get_rng_state()
                             if torch.backends.mps.is_available() else None),
                ), state_path)

    assert opt_step == cfg['max_updates']
    assert batch_pos == 0 and epoch_order is None
    assert len(eval_log) == cfg['max_updates'] // cfg['eval_every_updates']
    return dict(
        eval_log=eval_log, loss_hist=loss_hist,
        n_params=count_params(net), model_channels=MODEL_CHANNELS,
        n_train=n_train, variant=variant, c=c_val,
        minutes=(time.time() - t_start) / 60,
    )


## Run all arms

Sharding is strided across the 45 conceptual arms. Completed full-run $n=2$ artifacts are read
directly from the source directory and are never copied or overwritten.


In [ ]:
NOTE = (
    'SongUNet dataset-size sweep; exact architecture/data/rescaling/regularizers/sampler from '
    'songunet_covariance_tikhonov.ipynb. Nested n_train in [2,4,8,16,32]. Fixed 100000 '
    'optimizer updates, batch size 1, training seed 0. Arms: unregularized, isotropic and '
    'analytic population-covariance Tikhonov at c=[.003,.01,.03,.1]. Ring metric uses '
    'exclude_nn=True, mean_of_ratios, fixed reference RNG, and per-n held-out neutral baselines.'
)
N_EXPECT = CFG['max_updates'] // CFG['eval_every_updates']

SHARD = int(os.environ.get('ARM_SHARD', os.environ.get('SLURM_ARRAY_TASK_ID', 0)))
NSHARDS = int(os.environ.get('ARM_NSHARDS', os.environ.get('SLURM_ARRAY_TASK_COUNT', 1)))
assert 0 <= SHARD < NSHARDS
WRITE_FIGURES = (NSHARDS == 1) or (
    os.environ.get('WRITE_FIGURES', '0').strip().lower() in ('1', 'true', 'yes'))
my_arms = ARMS[SHARD::NSHARDS]
print(f'shard {SHARD} of {NSHARDS}: {len(my_arms)} of {len(ARMS)} conceptual arms')
print(f'write figure files: {WRITE_FIGURES}')


def reusable_result_path(arm):
    target = arm_result_path(arm['n_train'], arm['variant'], arm['c'])
    if os.path.exists(target):
        return target
    if REUSE_N2 and arm['n_train'] == 2:
        source = source_n2_path(arm['variant'], arm['c'])
        if os.path.exists(source):
            return source
    return None

for arm in my_arms:
    existing = reusable_result_path(arm)
    if existing is not None:
        prev = torch.load(existing, map_location='cpu', weights_only=False)
        if arm['n_train'] == 2 and existing.startswith(SOURCE_N2_DIR):
            assert prev['eval_log'][-1]['opt_step'] == CFG['max_updates']
            assert prev['model_channels'] == MODEL_CHANNELS
            assert torch.equal(prev['data'], data_all[:2])
            assert torch.equal(prev['lam_pop'], lam_pop)
        if len(prev['eval_log']) >= N_EXPECT:
            print(f'=== {arm["label"]} complete; reusing {existing} ===', flush=True)
            continue

    print(f'=== {arm["label"]} ===', flush=True)
    res = run_arm(arm, CFG, DEVICE, resume=True)
    res.update(
        cfg={k: v for k, v in CFG.items()}, data=data_all[:arm['n_train']].cpu(),
        note=NOTE, gmm_gate=GMM_GATES[arm['n_train']], D_minus=D_MINUS,
        scale=SCALE, sigma_max=CFG['sigma_max'], lam_pop=lam_pop.cpu(),
        inv_lam_pop=inv_lam_pop.cpu(), c_values=C_VALUES,
        null_rel_threshold=NULL_REL_THRESHOLD, neutral=neutral[arm['n_train']],
    )
    p = arm_result_path(arm['n_train'], arm['variant'], arm['c'])
    torch.save(res, p)
    print(f'  done in {res["minutes"]:.1f} min -> {p}', flush=True)

print('\nArm status:')
for arm in ARMS:
    p = reusable_result_path(arm)
    status = 'COMPLETE' if p is not None else 'MISSING'
    print(f'  {arm["label"]:>32s}  {status:>8s}  {p or ""}')


## Results

Analysis cells tolerate partial shards: figures appear as soon as the needed arms are present.


In [ ]:
def load_all_arms():
    out = {}
    for arm in ARMS:
        p = reusable_result_path(arm)
        if p is not None:
            r = torch.load(p, map_location='cpu', weights_only=False)
            if r.get('eval_log'):
                out[(arm['n_train'], arm['variant'], arm['c'])] = r
    return out

all_runs = load_all_arms()
print(f'Loaded {len(all_runs)}/{len(ARMS)} arms')
for key, r in sorted(all_runs.items()):
    n_train, variant, c_val = key
    last = r['eval_log'][-1]
    raw_c = last['coarse_score']
    raw_f = last['fine_score']
    # Old n=2 artifacts predate stored normalized fields; n=2 references are deterministic.
    c_norm = last.get('coarse_over_neutral', raw_c / neutral[n_train]['coarse_score'])
    f_norm = last.get('fine_over_neutral', raw_f / neutral[n_train]['fine_score'])
    print(f'n={n_train:>2} {variant:>14s} c={c_val:<6g} update={last.get("opt_step", 0):>7,} '
          f'coarse/base={c_norm:.4f} fine/base={f_norm:.4f}')


### Main result: scale selectivity versus dataset size

Each column fixes `c`; rows show coarse and fine bands. The horizontal line at 1 is the
per-dataset-size neutral baseline, not a universal raw-score threshold.


In [ ]:
if all((n, 'gate', 0.0) in all_runs for n in N_TRAIN_VALUES):
    fig, axes = plt.subplots(2, len(C_VALUES), figsize=(4.1*len(C_VALUES), 7.2), sharex=True)
    axes = np.asarray(axes).reshape(2, len(C_VALUES))
    for col, c_val in enumerate(C_VALUES):
        for row, band in enumerate(['coarse', 'fine']):
            ax = axes[row, col]
            for variant, color, marker, label in [
                ('gate', 'tab:red', 'o', 'unregularized'),
                ('isotropic', 'tab:gray', 's', 'isotropic'),
                ('cov_population', 'tab:green', '^', 'analytic covariance'),
            ]:
                xs, ys = [], []
                for n_train in N_TRAIN_VALUES:
                    key = (n_train, variant, 0.0 if variant == 'gate' else c_val)
                    if key not in all_runs:
                        continue
                    last = all_runs[key]['eval_log'][-1]
                    raw = last[f'{band}_score']
                    val = last.get(f'{band}_over_neutral',
                                   raw / neutral[n_train][f'{band}_score'])
                    xs.append(n_train); ys.append(val)
                if xs:
                    ax.plot(xs, ys, color=color, marker=marker, lw=1.7, label=label)
            ax.axhline(1.0, color='black', ls=':', lw=1)
            ax.set_xscale('log', base=2)
            ax.set_xticks(N_TRAIN_VALUES); ax.set_xticklabels(N_TRAIN_VALUES)
            ax.set_title(f'{band}, c={c_val:g}')
            ax.set_xlabel('n_train')
            if col == 0:
                ax.set_ylabel('raw band score / per-n neutral')
            if row == 0 and col == len(C_VALUES)-1:
                ax.legend(fontsize=8)
    fig.suptitle('SongUNet covariance Tikhonov across dataset size\n'
                 'fixed 100,000 optimizer updates per arm', y=1.02)
    plt.tight_layout()
    f = os.path.join(fig_dir, f'songunet_cov_tikhonov_dataset_size_{RUN_TAG}.png')
    if WRITE_FIGURES:
        plt.savefig(f, dpi=180, bbox_inches='tight')
        print(f'saved {f}')
    plt.show()
else:
    print('Main figure awaits all unregularized arms.')


### Covariance weighting across $c$, resolved by wavenumber

Each panel fixes the dataset size and compares only the four analytic-covariance arms.
Values are divided by the per-$n$ held-out neutral profile; lower values indicate stronger
memorization at that wavenumber.


In [ ]:
kc = ctx.k_centers.cpu().numpy()
profile_mask = (kc >= 0.5) & (kc <= 32)
profile_colors = plt.cm.viridis(np.linspace(0.12, 0.88, len(C_VALUES)))
profiles = {}
for n_train in N_TRAIN_VALUES:
    for c_val in C_VALUES:
        key = (n_train, 'cov_population', c_val)
        if key in all_runs:
            last = all_runs[key]['eval_log'][-1]
            neu = last.get('neutral_mean_ratio', neutral[n_train]['mean_ratio']).numpy()
            profiles[(n_train, c_val)] = last['mean_ratio'].numpy() / neu

profile_max = max(np.nanmax(v[profile_mask]) for v in profiles.values())
fig, axes = plt.subplots(2, 3, figsize=(13.2, 7.4), sharex=True, sharey=True)
axes = np.asarray(axes).ravel()
for ax, n_train in zip(axes, N_TRAIN_VALUES):
    for c_val, color in zip(C_VALUES, profile_colors):
        y = profiles.get((n_train, c_val))
        if y is not None:
            ax.plot(kc[profile_mask], y[profile_mask], color=color, lw=2,
                    label=fr'$c={c_val:g}$')
    ax.axhline(1.0, color='black', ls=':', lw=1, label='neutral')
    for edge in [4, 10, 18, 32]:
        ax.axvline(edge, color='0.82', ls='--', lw=.8, zorder=0)
    ax.set_xlim(0.5, 32); ax.set_ylim(0, 1.05*profile_max)
    ax.set_title(fr'$n_{{\mathrm{{train}}}}={n_train}$')
    ax.set_xlabel('wavenumber $k$')
    ax.grid(axis='y', alpha=.18)
for ax in axes[::3]:
    ax.set_ylabel('ring ratio / neutral')
axes[-1].axis('off')
handles, labels = axes[0].get_legend_handles_labels()
fig.legend(handles, labels, loc='lower right', bbox_to_anchor=(.91, .16), frameon=False)
fig.suptitle('Analytic covariance weighting: memorization by wavenumber and strength')
plt.tight_layout(rect=(0, 0, 1, .96))
f = os.path.join(fig_dir, f'songunet_cov_tikhonov_covariance_per_k_by_c_{RUN_TAG}.png')
if WRITE_FIGURES:
    plt.savefig(f, dpi=180, bbox_inches='tight')
    print(f'saved {f}')
plt.show()


### Same-$c$ comparison of covariance and isotropic weighting

The headline figure compares direct neutral-normalized coarse- and fine-band scores at
$n_{\mathrm{train}}=2$ and the same $c=0.1$. This avoids selecting different strengths or
introducing a ratio of band-score ratios. The full-grid method-ratio figure is retained only
as a secondary diagnostic. The final figure gives absolute scores for all four bands and all
dataset sizes at $c=0.1$, including the unregularized control.


In [ ]:
def normalized_band_score(n_train, variant, c_val, band):
    last = all_runs[(n_train, variant, c_val)]['eval_log'][-1]
    return last.get(f'{band}_over_neutral',
                    last[f'{band}_score'] / neutral[n_train][f'{band}_score'])

band_order = ['coarse', 'mid1', 'mid2', 'fine']
band_titles = {'coarse': 'Coarse: 0.5 < k < 4', 'mid1': 'Mid 1: 4 < k < 10',
               'mid2': 'Mid 2: 10 < k < 18', 'fine': 'Fine: 18 < k < 32'}
c_colors = plt.cm.viridis(np.linspace(0.12, 0.88, len(C_VALUES)))

# Headline same-strength comparison: direct scores, not a ratio of ratios.
STORY_C = 0.1
story_bands = ['coarse', 'fine']
story_x = np.arange(len(story_bands), dtype=float)
story_styles = [
    ('isotropic', 'tab:gray', 's', 'isotropic'),
    ('cov_population', 'tab:green', '^', 'analytic covariance'),
]
story_scores = {
    variant: np.array([normalized_band_score(2, variant, STORY_C, band)
                       for band in story_bands])
    for variant, _, _, _ in story_styles
}

fig, ax = plt.subplots(figsize=(7.4, 4.8))
ax.axvspan(-.38, .38, color='tab:blue', alpha=.055, zorder=0)
ax.axvspan(.62, 1.38, color='#d9a400', alpha=.07, zorder=0)
for offset, (variant, color, marker, label) in zip([-.025, .025], story_styles):
    ax.plot(story_x + offset, story_scores[variant], color=color, marker=marker,
            ms=9, lw=2.4, label=label, zorder=3)
    ax.annotate(f'{story_scores[variant][0]:.3f}',
                (story_x[0] + offset, story_scores[variant][0]), xytext=(0, 8),
                textcoords='offset points', ha='center', color=color, fontsize=9)
ax.axhline(1.0, color='black', ls=':', lw=1.4, label='neutral', zorder=1)
coarse_iso = story_scores['isotropic'][0]
coarse_cov = story_scores['cov_population'][0]
coarse_reduction = 100 * (coarse_iso - coarse_cov) / coarse_iso
ax.annotate('', xy=(.17, coarse_cov), xytext=(.17, coarse_iso),
            arrowprops=dict(arrowstyle='<->', color='tab:green', lw=1.5))
ax.text(.20, np.sqrt(coarse_iso * coarse_cov),
        f'{coarse_reduction:.0f}% lower\ncoarse score', color='tab:green',
        va='center', fontsize=9)
ax.text(.88, .91, f"{story_scores['isotropic'][1]:.3f}",
        ha='right', va='top', fontsize=9, color='tab:gray')
ax.text(1.12, .91, f"{story_scores['cov_population'][1]:.3f}",
        ha='left', va='top', fontsize=9, color='tab:green')
ax.text(1.0, .80, 'both near neutral', ha='center', va='top', fontsize=9, color='0.3')
ax.set_yscale('log')
ax.set_ylim(.08, 1.12); ax.set_xlim(-.42, 1.42)
ax.set_xticks(story_x)
ax.set_xticklabels(['Coarse\n$0.5 < k < 4$', 'Fine\n$18 < k < 32$'])
ax.set_yticks([.1, .2, .5, 1.0]); ax.set_yticklabels(['0.1', '0.2', '0.5', '1.0'])
ax.set_ylabel('Band score')
ax.set_title('Band scores for isotropic and covariance-weighted Tikhonov regularization\n'
             r'$n_{\mathrm{train}}=2$, $c=0.1$')
ax.grid(axis='y', which='both', alpha=.18)
ax.legend(loc='center right', bbox_to_anchor=(.98, .47), frameon=False)
plt.tight_layout()
f = os.path.join(fig_dir,
                 f'songunet_cov_tikhonov_n2_c0p1_direct_band_scores_{RUN_TAG}.png')
if WRITE_FIGURES:
    plt.savefig(f, dpi=220, bbox_inches='tight')
    print(f'saved {f}')
plt.show()

# Full-grid, equal-c effect: no post-hoc c selection.
fig, axes = plt.subplots(2, 2, figsize=(10.8, 7.8), sharex=True, sharey=True)
for ax, band in zip(axes.ravel(), band_order):
    for c_val, color in zip(C_VALUES, c_colors):
        ratio = [normalized_band_score(n, 'isotropic', c_val, band) /
                 normalized_band_score(n, 'cov_population', c_val, band)
                 for n in N_TRAIN_VALUES]
        ax.plot(N_TRAIN_VALUES, ratio, color=color, marker='o', ms=5, lw=1.8,
                label=fr'$c={c_val:g}$')
    ax.axhline(1.0, color='black', ls=':', lw=1.2)
    ax.axhspan(1.0, 2.5, color='tab:green', alpha=.035, zorder=0)
    ax.set_xscale('log', base=2); ax.set_yscale('log')
    ax.set_xticks(N_TRAIN_VALUES); ax.set_xticklabels(N_TRAIN_VALUES)
    ax.set_ylim(.8, 2.5)
    ax.set_title(band_titles[band])
    ax.set_xlabel(r'$n_{\mathrm{train}}$')
    ax.set_ylabel(r'$R_{\mathrm{iso}} / R_{\mathrm{cov}}$')
    ax.grid(axis='y', which='both', alpha=.18)
axes[0, 0].text(2.1, 2.18, 'covariance retains more memorization',
                color='tab:green', fontsize=8)
axes[0, 0].text(2.1, .84, 'isotropic retains more', color='0.35', fontsize=8)
handles, labels = axes[0, 0].get_legend_handles_labels()
fig.legend(handles, labels, loc='upper center', bbox_to_anchor=(.5, .955),
           ncol=len(C_VALUES), frameon=False)
fig.suptitle('Same-strength comparison across the full dataset-size sweep', y=.995)
plt.tight_layout(rect=(0, 0, 1, .91))
f = os.path.join(fig_dir, f'songunet_cov_tikhonov_same_c_method_ratio_{RUN_TAG}.png')
if WRITE_FIGURES:
    plt.savefig(f, dpi=180, bbox_inches='tight')
    print(f'saved {f}')
plt.show()

# Screenshot-style paired bars: same c within every isotropic/covariance pair.
fig, axes = plt.subplots(2, 3, figsize=(14.2, 8.2), sharey=True)
axes = axes.ravel()
x = np.arange(len(C_VALUES))
bar_width = .36
legend_handles = None
for ax, n_train in zip(axes, N_TRAIN_VALUES):
    coarse_iso = [normalized_band_score(n_train, 'isotropic', c, 'coarse')
                  for c in C_VALUES]
    coarse_cov = [normalized_band_score(n_train, 'cov_population', c, 'coarse')
                  for c in C_VALUES]
    bars_iso = ax.bar(x-bar_width/2, coarse_iso, bar_width, color='tab:gray',
                      label='isotropic')
    bars_cov = ax.bar(x+bar_width/2, coarse_cov, bar_width, color='tab:green',
                      label='analytic covariance')
    ax.bar_label(bars_iso, fmt='%.3f', padding=2, fontsize=7, rotation=90)
    ax.bar_label(bars_cov, fmt='%.3f', padding=2, fontsize=7, rotation=90)
    tick_labels = [fr'$c={c:g}$' for c in C_VALUES]
    ax.set_xticks(x); ax.set_xticklabels(tick_labels, fontsize=7)
    ax.axhline(1.0, color='black', ls=':', lw=1, label='neutral')
    ax.set_ylim(0, 1.08)
    ax.set_title(fr'$n_{{\mathrm{{train}}}}={n_train}$')
    ax.set_ylabel('coarse score / neutral')
    ax.grid(axis='y', alpha=.18)
    legend_handles = (bars_iso[0], bars_cov[0])
axes[-1].axis('off')
axes[-1].legend(legend_handles, ['isotropic', 'analytic covariance'],
                loc='center', frameon=False, fontsize=11)
axes[-1].text(.5, .32, 'Each pair uses the same c.\nLower coarse score = more memorization.\nFine and mid-band scores are shown\nin the four-band companion figure.',
              transform=axes[-1].transAxes, ha='center', va='center', fontsize=10)
fig.suptitle('Same-strength coarse memorization across dataset size')
plt.tight_layout(rect=(0, 0, 1, .95))
f = os.path.join(fig_dir, f'songunet_cov_tikhonov_same_c_grouped_bars_{RUN_TAG}.png')
if WRITE_FIGURES:
    plt.savefig(f, dpi=180, bbox_inches='tight')
    print(f'saved {f}')
plt.show()

# Absolute context at the shared c where fine suppression is already matched at n=2.
assert STORY_C in C_VALUES
fig, axes = plt.subplots(2, 2, figsize=(10.8, 7.8), sharex=True, sharey=True)
for ax, band in zip(axes.ravel(), band_order):
    for variant, color, marker, label in [
        ('gate', 'tab:red', 'o', 'unregularized'),
        ('isotropic', 'tab:gray', 's', 'isotropic'),
        ('cov_population', 'tab:green', '^', 'analytic covariance'),
    ]:
        c_val = 0.0 if variant == 'gate' else STORY_C
        ys = [normalized_band_score(n, variant, c_val, band)
              for n in N_TRAIN_VALUES]
        ax.plot(N_TRAIN_VALUES, ys, color=color, marker=marker, ms=6, lw=2, label=label)
    ax.axhline(1.0, color='black', ls=':', lw=1.2, label='neutral')
    ax.set_xscale('log', base=2); ax.set_yscale('log')
    ax.set_xticks(N_TRAIN_VALUES); ax.set_xticklabels(N_TRAIN_VALUES)
    ax.set_ylim(.045, 1.08)
    ax.set_title(band_titles[band])
    ax.set_xlabel(r'$n_{\mathrm{train}}$')
    ax.set_ylabel('band score / neutral')
    ax.grid(axis='y', which='both', alpha=.18)
handles, labels = axes[0, 0].get_legend_handles_labels()
fig.legend(handles, labels, loc='upper center', bbox_to_anchor=(.5, .955),
           ncol=4, frameon=False)
fig.suptitle(fr'Dataset-size transition at the same strength $c={STORY_C:g}$', y=.995)
plt.tight_layout(rect=(0, 0, 1, .91))
f = os.path.join(fig_dir, f'songunet_cov_tikhonov_c0p1_all_bands_log_{RUN_TAG}.png')
if WRITE_FIGURES:
    plt.savefig(f, dpi=180, bbox_inches='tight')
    print(f'saved {f}')
plt.show()


### Final summary table


In [ ]:
print(f'{"n":>3s} {"variant":>15s} {"c":>7s} {"updates":>8s} {"epochs":>8s} '
      f'{"frac":>7s} {"coarse":>9s} {"c/base":>9s} {"fine":>9s} {"f/base":>9s}')
print('-' * 100)
for (n_train, variant, c_val), r in sorted(all_runs.items()):
    last = r['eval_log'][-1]
    c_norm = last['coarse_score'] / neutral[n_train]['coarse_score']
    f_norm = last['fine_score'] / neutral[n_train]['fine_score']
    print(f'{n_train:>3d} {variant:>15s} {c_val:>7g} {last.get("opt_step", 0):>8d} '
          f'{last.get("epoch", 0):>8d} {last["fraction"]:>7.3f} '
          f'{last["coarse_score"]:>9.4f} {c_norm:>9.4f} '
          f'{last["fine_score"]:>9.4f} {f_norm:>9.4f}')


## Interpretation guardrails

- The primary comparison is analytic covariance versus isotropic at the same dataset size and
  the same budget-normalized $c$. An unequal-$c$, matched-outcome comparison is a secondary
  efficiency diagnostic and must be labelled as such. Do not compare raw scores vertically
  across $n$ without the neutral normalization.
- A value near the neutral line is a null/generalization result; it is not evidence that the
  regularizer “fixed” memorization.
- This notebook isolates dataset size at a fixed update budget. It does **not** answer the
  separate equal-epoch question. If that question becomes necessary, extend selected arms in a
  second figure and label the changed update budget explicitly.
- One training seed is retained to isolate the requested dataset-size axis. A multi-seed rerun is
  a separate follow-up, as Prof. Baptista indicated.
